# 05 - Fold Verification

**Goal:** Verify that fold sizes are balanced (ratio ≤ 2×) in the cross-validation setup.

**Critical Decision Gate:** If fold sizes are uneven, the entire CV structure needs fixing before proceeding to Phase 2.

**Tasks:**
- Load train/CV data (up to 2025-09-30 23:59:59)
- Instantiate PurgedWalkForward(n_splits=5, test_size=0.15, embargo=48)
- Loop through folds and log: fold number, train rows, test rows, date range, duration in months
- Calculate max/min test rows and ratio
- Determine if fold sizes are balanced (ratio ≤ 2×) or uneven
- Create decision document

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path

# Import the PurgedWalkForward CV splitter
from src.cv import PurgedWalkForward

print('Dependencies imported successfully.')

## 1. Load Data

In [ ]:
# Load data
DATA_PATH = Path('../data/DataCollector_EURUSD_M5_20230101_220400.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Total rows in data: {len(df):,}')
print(f'Date range: {df.timestamp.min()} to {df.timestamp.max()}')

## 2. Filter to Train/CV Window

In [ ]:
# Filter to train/CV window (must match what walk-forward used)
train_cv_end = pd.Timestamp('2025-09-30 23:59:59')
df_cv = df[df.timestamp <= train_cv_end].reset_index(drop=True)

print(f'Train/CV rows: {len(df_cv):,}')
print(f'Date range: {df_cv.timestamp.min()} to {df_cv.timestamp.max()}')
print(f'Duration: {(df_cv.timestamp.max() - df_cv.timestamp.min()).days / 30.44:.1f} months')

## 3. Initialize Cross-Validation Splitter

In [ ]:
# Initialize PurgedWalkForward with exact spec
cv = PurgedWalkForward(
    n_splits=5,
    embargo_bars=48,
    test_size=0.15
)

print(f'CV Setup:')
print(f'  n_splits: {cv.n_splits}')
print(f'  embargo_bars: {cv.embargo_bars}')
print(f'  test_size: {cv.test_size}')

## 4. Run Fold Verification

In [ ]:
print('='*70)
print('FOLD SIZE VERIFICATION')
print('='*70)

fold_info = []

# Use df_cv as the data to split
for fold_num, (train_idx, test_idx) in enumerate(cv.split(df_cv), 1):
    test_rows = len(test_idx)
    train_rows = len(train_idx)
    
    # Get date range for test fold
    test_start = df_cv['timestamp'].iloc[test_idx[0]]
    test_end = df_cv['timestamp'].iloc[test_idx[-1]]
    
    # Calculate duration in months
    months = (test_end - test_start).days / 30.44
    
    fold_info.append({
        'fold': fold_num,
        'train_rows': train_rows,
        'test_rows': test_rows,
        'test_start': test_start.date(),
        'test_end': test_end.date(),
        'months': round(months, 1),
    })
    
    print(f'Fold {fold_num}: train={train_rows:>7,}  test={test_rows:>6,}  '
          f'period={test_start.date()} → {test_end.date()} ({months:.1f} months)')

# Create DataFrame
fi = pd.DataFrame(fold_info)
print('\n' + '='*70)
print('FOLD SIZES SUMMARY')
print('='*70)
print(fi.to_string(index=False))

## 5. Calculate Fold Size Ratio

In [ ]:
max_test = fi['test_rows'].max()
min_test = fi['test_rows'].min()
ratio = max_test / min_test

print(f'\nLargest fold:  {max_test:,} bars')
print(f'Smallest fold: {min_test:,} bars')
print(f'Ratio:         {ratio:.2f}×')
print()

# Decision
if ratio > 2.0:
    decision = 'UNEVEN'
    print('⚠️  WARNING: Fold sizes are uneven (ratio > 2×).')
    print('    The largest fold dominates any averaged metric.')
    print('    DECISION: CV structure needs fixing before proceeding.')
else:
    decision = 'BALANCED'
    print('✅ Fold sizes are reasonably balanced (ratio ≤ 2×).')
    print('   DECISION: Proceed to Phase 2.')

## 6. Export Fold Verification Results

In [ ]:
# Save fold info CSV
output_path = Path('../outputs/fold_verification.csv')
output_path.parent.mkdir(exist_ok=True, parents=True)
fi.to_csv(output_path, index=False)
print(f'Fold verification data saved to: {output_path}')

# Print summary
print(f'\nFold Ratio: {ratio:.2f}×')
print(f'Status: {decision}')

## Summary

**Fold Verification Complete**

Results are exported to outputs/fold_verification.csv